In [ ]:
"""
GeneTraceAI — Layer 2 Core Score
=================================
Grain   : gene × cell-line keyed (model_id [UPPER ACH-], ensg_id [bare, no version])
Inputs  : depmap_expr (transcriptomics, log2 TPM), proteomics (log-ratio)
Output  : core_score — a RANKING score, not a calibrated probability (see below)

CALIBRATION STATUS — read this before using the score:
  Every coefficient (percentile bins, Noisy-OR combination, layer weights)
  is an uncalibrated free parameter. Validation against CRISPR/GDSC anchor
  pairs (BRAF/A375, KRAS/HCT116, held-out TSG-fusion arm) has NOT been run.
  This notebook produces a working, inspectable score with honest
  normalisation — not a calibrated one.

DESIGN DECISION — ranking vs probability:
  core_score is a RANKING, not a probability.
  Reason: Noisy-OR assumes conditional independence between layers.
  mRNA and protein levels are correlated (literature: ρ ≈ 0.58, Nusinow 2020).
  Under dependence, Noisy-OR inflates scores where both layers agree —
  but the inflation is monotonic, so gene ordering within a cell line
  is preserved (van Krieken 2024 notes this; Trick 2026 confirms for
  multi-omics fusion contexts). A calibrated probability would require
  correlation correction or a held-out regression, which needs the
  validation harness first. Until that exists, core_score should be
  used for ranking only (top-N genes per cell line, comparative
  across cell lines for the same gene) — not as a literal P(alteration).

WHAT IS OUT OF SCOPE HERE (later layers):
  - Mutation / fusion flags (Layer 3, left-joined after scoring)
  - Signature discount — MSI/CIN multiplicative form uncited; deferred
  - miRNA, metabolomics — no gene axis; excluded from gene-level scoring
  - Similarity propagation (Layer 4)

ENGINEERING RULES (applied throughout, all motivated by past bugs):
  1. .str.lower() model_id before every join (depmap_profiles emits lowercase)
  2. .str.split(".").str[0] on ensg_id before every join (version suffixes silently fail)
  3. assert len(out)==len(left) after every left join (catches fan-out)
  4. Report match rates after every join — low rate = key mismatch, not absence
  5. Strip pandas metadata when reading list-typed parquets

Author: Chaithali
Date  : 2026-07-19
"""

import sys, os, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT          = Path(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
DATA_CLEAN    = ROOT / "data" / "parquet" / "data_clean"
CLEANED_TRACK = ROOT / "cleaned_track_data"
REF_DIR       = ROOT / "reference"
PIPELINE_OUT  = Path(os.getcwd()) / "outputs"
PIPELINE_OUT.mkdir(exist_ok=True)

for label, p in [("DATA_CLEAN", DATA_CLEAN), ("CLEANED_TRACK", CLEANED_TRACK),
                  ("REF_DIR", REF_DIR), ("PIPELINE_OUT", PIPELINE_OUT)]:
    print(f"{label}: {p}  exists={p.is_dir()}")

In [ ]:
# ── Cell 2 — Gene universe filter ─────────────────────────────────────────


gene_lookup = pd.read_parquet(REF_DIR / "gene_lookup.parquet")

universe = gene_lookup[
    (gene_lookup["biotype"] == "protein_coding") &
    (gene_lookup["hgnc_status"] == "Approved")
][["ensg_id", "hgnc_symbol", "uniprot_ids"]].copy()

universe["ensg_id"] = universe["ensg_id"].astype("string").str.split(".").str[0].str.lower()

valid_ensg = set(universe["ensg_id"])

print(f"Gene universe: {len(universe):,} genes (protein_coding + Approved)")
print(f"Sample ensg_id: {universe['ensg_id'].head(3).tolist()}")
print(f"Sample uniprot_ids: {universe['uniprot_ids'].head(3).tolist()}")

In [ ]:
# ── Cell 3 — Build UniProt → ENSG lookup ──────────────────────────────────
# Proteomics columns are lowercase UniProt accessions.
# gene_lookup.uniprot_ids is a single accession per row (string, not list).
# Lowercase both sides before joining.

uniprot_to_ensg = (
    universe[["ensg_id", "uniprot_ids"]]
    .dropna(subset=["uniprot_ids"])
    .assign(uniprot_id=lambda x: x["uniprot_ids"].str.strip().str.lower())
    .drop_duplicates(subset=["uniprot_id"])
    [["uniprot_id", "ensg_id"]]
)

print(f"UniProt→ENSG lookup: {len(uniprot_to_ensg):,} mappings")
print(f"Sample: {uniprot_to_ensg.head(3).to_string(index=False)}")

In [ ]:
# -- Cell 3b - PATCH: UniProt->ENSG mapping, isoform- and multi-accession-aware --
# Supersedes the `uniprot_to_ensg` built in Cell 3. Kept as a separate cell so the
# original derivation stays visible and auditable; Cell 5 is unchanged and picks
# this object up because notebooks execute top-to-bottom.
#
# Cell 3 keyed the join on an exact lowercase string match against
# gene_lookup.uniprot_ids. That silently dropped protein data two ways:
#
#   1. ISOFORM SUFFIXES. The proteomics matrix identifies 1,307 of its 10,129
#      columns by isoform accession (MUC1 is 'p15941-2'), while
#      gene_lookup.uniprot_ids only ever holds the canonical accession -- 0 of
#      19,213 rows contain a '-'. Those columns never matched, so the protein
#      layer for those genes vanished with no error: the gene simply scored as
#      RNA-only (n_layers=1) and nothing recorded that a layer had been lost.
#
#   2. MULTI-ACCESSION GENES. 40 gene_lookup rows hold several '|'-separated
#      accessions. The whole string was lowercased and used as a single key, so
#      none of those genes could ever match a single-accession column.
#
# Effect: genes carrying a protein layer 8,564 -> 9,672 (+1,108, +12.9%).
# 268 matrix columns remain unmapped -- their accession is not in the
# protein_coding + Approved universe at all. That is a genuine data gap and is
# reported below rather than silently absorbed.
#
# COLLISIONS: 182 genes end up with more than one proteomics column (canonical
# plus one or more isoforms). Cell 5 pivots with aggfunc="mean", so these are
# averaged into one per-gene value -- the same rule that already applied to any
# gene with duplicate accessions. Averaging isoform intensities of the same
# protein is the conservative choice; summing would inflate genes that simply
# happen to have more isoforms quantified.

import re
import pyarrow.parquet as pq

# (1) Split multi-accession strings and explode to one accession per row.
_acc = (
    universe[["ensg_id", "uniprot_ids"]]
    .dropna(subset=["uniprot_ids"])
    .assign(uniprot_id=lambda x: x["uniprot_ids"].astype("string")
                                  .str.lower().str.split(r"[|;,]"))
    .explode("uniprot_id")
)
_acc["uniprot_id"] = _acc["uniprot_id"].str.strip()
_acc = _acc[_acc["uniprot_id"].str.len() > 0]

uniprot_to_ensg = _acc.drop_duplicates(subset=["uniprot_id"])[["uniprot_id", "ensg_id"]]
_canonical = dict(zip(uniprot_to_ensg["uniprot_id"], uniprot_to_ensg["ensg_id"]))

# (2) Add alias keys for isoform-suffixed columns that actually occur in the
#     proteomics matrix, so Cell 5's exact-match merge resolves them. Schema-only
#     read -- no data is loaded here.
_prot_cols = [c for c in pq.read_schema(CLEANED_TRACK / "proteomics.parquet").names
              if c not in ("depmap_id", "model_id")]

_alias_rows, _unmappable = [], []
for _c in _prot_cols:
    _k = _c.strip().lower()
    if _k in _canonical:
        continue
    _base = re.sub(r"-\d+$", "", _k)
    if _base in _canonical:
        _alias_rows.append({"uniprot_id": _k, "ensg_id": _canonical[_base]})
    else:
        _unmappable.append(_k)

if _alias_rows:
    uniprot_to_ensg = pd.concat(
        [uniprot_to_ensg, pd.DataFrame(_alias_rows)], ignore_index=True
    ).drop_duplicates(subset=["uniprot_id"])

_covered = {_canonical[c.strip().lower()] for c in _prot_cols
            if c.strip().lower() in _canonical}
_covered |= {r["ensg_id"] for r in _alias_rows}

print(f"UniProt->ENSG lookup (patched): {len(uniprot_to_ensg):,} mappings")
print(f"  isoform alias keys added      : {len(_alias_rows):,}")
print(f"  proteomics columns unmappable : {len(_unmappable):,} "
      f"(accession absent from the gene universe -- genuine data gap)")
print(f"  genes with a protein layer    : {len(_covered):,} / {len(_prot_cols):,} columns")

assert uniprot_to_ensg["uniprot_id"].is_unique, "uniprot_id keys must be unique before the Cell 5 merge"


In [ ]:
# ── Cell 4 — Load depmap_expr + resolve profile → model_id ────────────────
# depmap_expr: shape (1495, 53961), index = lowercase pr-... profile IDs
# depmap_profiles: profileid, modelid, datatype — filter datatype=='rna'
#   modelid is LOWERCASE — must .str.lower() before any join.

# --- Load profiles, filter to RNA ---
prof = pd.read_parquet(REF_DIR / "depmap_profiles.parquet")
rna_prof = (
    prof[prof["datatype"] == "rna"][["profileid", "modelid"]]
    .copy()
)
rna_prof["model_id"] = rna_prof["modelid"].str.lower()   # rule 1
rna_prof = rna_prof.drop(columns=["modelid"])

print(f"RNA profiles: {len(rna_prof):,} rows")
print(f"profileid sample: {rna_prof['profileid'].head(3).tolist()}")
print(f"model_id sample:  {rna_prof['model_id'].head(3).tolist()}")
assert rna_prof["profileid"].duplicated().sum() == 0, "Duplicate profileids in RNA profiles"

# --- Load expression matrix ---
expr_raw = pd.read_parquet(DATA_CLEAN / "depmap_expr_clean.parquet")
expr_raw.index.name = "profileid"

print(f"\nexpression matrix shape: {expr_raw.shape}")
print(f"index sample: {expr_raw.index[:3].tolist()}")
print(f"col sample: {expr_raw.columns[:5].tolist()}")

# --- Join profile → model_id ---
# Reset index so profileid becomes a join column
expr = expr_raw.reset_index()                            # profileid now a column
before = len(expr)
expr = expr.merge(rna_prof, on="profileid", how="inner")  # inner: only matched profiles
print(f"\nAfter profile join: {len(expr):,} rows (was {before:,})")
print(f"Match rate: {len(expr)/before*100:.1f}%")
assert len(expr) == before, f"Fan-out detected: {before} → {len(expr)} rows"

# Move model_id to front, drop profileid
expr = expr.drop(columns=["profileid"]).set_index("model_id")
# Uppercase column ENSG IDs (rule 2 — version strip not needed, already bare)
expr.columns = expr.columns.str.lower()

print(f"Expression shape after join: {expr.shape}")
print(f"model_id sample: {expr.index[:3].tolist()}")
print(f"col sample: {expr.columns[:5].tolist()}")

# --- Filter to gene universe ---
before_cols = expr.shape[1]
expr = expr[[c for c in expr.columns if c in valid_ensg]]
print(f"\nGene universe filter: {before_cols:,} → {expr.shape[1]:,} columns")

In [ ]:
# ── Cell 5 — Load proteomics + map UniProt → ENSG ─────────────────────────
# File: cleaned_track_data/proteomics.parquet
#   Shape: (375, 10130), key col: depmap_id (ACH- uppercase already)
#   Feature cols: lowercase UniProt accessions
#   Already Track-D filtered (>70% missing cols dropped)
# Strategy: melt to long → join uniprot_to_ensg → filter to universe → pivot wide

prot_raw = pd.read_parquet(CLEANED_TRACK / "proteomics.parquet")

print(f"proteomics raw shape: {prot_raw.shape}")
print(f"key col ('depmap_id' present?): {'depmap_id' in prot_raw.columns}")
print(f"model_id present?: {'model_id' in prot_raw.columns}")
print(f"sample cols: {prot_raw.columns[:6].tolist()}")
key_col = "model_id" if "model_id" in prot_raw.columns else "depmap_id"
print(f"Using key col: '{key_col}'")
print(f"key sample: {prot_raw[key_col].head(3).tolist()}")

# --- Melt to long form ---
feature_cols = [c for c in prot_raw.columns if c != key_col]
prot_long = prot_raw.melt(
    id_vars=[key_col], var_name="uniprot_id", value_name="log_ratio"
).dropna(subset=["log_ratio"])
prot_long["model_id"] = prot_long[key_col].str.lower()    # rule 1
if key_col != "model_id":
    prot_long = prot_long.drop(columns=[key_col])

print(f"\nproteomics long (after melt+dropna): {len(prot_long):,} rows")

# --- Map UniProt → ENSG ---
before = len(prot_long)
prot_mapped = prot_long.merge(uniprot_to_ensg, on="uniprot_id", how="inner")
print(f"After UniProt→ENSG map: {len(prot_mapped):,} rows ({len(prot_mapped)/before*100:.1f}% of long rows mapped)")
print(f"Distinct ENSG IDs in proteomics: {prot_mapped['ensg_id'].nunique():,}")
print(f"Distinct model_ids:              {prot_mapped['model_id'].nunique():,}")

# Filter to gene universe
before = len(prot_mapped)
prot_mapped = prot_mapped[prot_mapped["ensg_id"].isin(valid_ensg)]
print(f"After universe filter: {len(prot_mapped):,} rows (was {before:,})")

# --- Pivot back to wide: model_id × ensg_id ---
prot = prot_mapped.pivot_table(
    index="model_id", columns="ensg_id", values="log_ratio", aggfunc="mean"
)
prot.columns.name = None

print(f"\nproteomics wide shape: {prot.shape}")
print(f"model_id sample: {prot.index[:3].tolist()}")
print(f"col sample: {prot.columns[:5].tolist()}")

In [ ]:
# ── Cell 6 — Input confirmation report ────────────────────────────────────
# Check: both inputs have model_id index, ENSG columns, overlapping genes + cells.

expr_models  = set(expr.index)
prot_models  = set(prot.index)
expr_genes   = set(expr.columns)
prot_genes   = set(prot.columns)
shared_models = expr_models & prot_models
shared_genes  = expr_genes  & prot_genes

print("=" * 60)
print("INPUT CONFIRMATION REPORT")
print("=" * 60)
print(f"\n[TRANSCRIPTOMICS — depmap_expr]")
print(f"  Shape          : {expr.shape}")
print(f"  Index type     : model_id (ACH-..., UPPER)")
print(f"  Column type    : ENSG (bare, UPPER)")
print(f"  Value type     : log2 TPM+1")
print(f"  Unique models  : {len(expr_models):,}")
print(f"  Unique genes   : {len(expr_genes):,}")

print(f"\n[PROTEOMICS — cleaned_track_data/proteomics.parquet → ENSG mapped]")
print(f"  Shape          : {prot.shape}")
print(f"  Index type     : model_id (ACH-..., UPPER)")
print(f"  Column type    : ENSG (bare, UPPER, via UniProt→ENSG join)")
print(f"  Value type     : log-ratio (CCLE proteomics, Nusinow 2020)")
print(f"  Unique models  : {len(prot_models):,}")
print(f"  Unique genes   : {len(prot_genes):,}")

print(f"\n[OVERLAP]")
print(f"  Shared cell lines : {len(shared_models):,} / expr {len(expr_models):,} / prot {len(prot_models):,}")
print(f"  Shared genes      : {len(shared_genes):,} / expr {len(expr_genes):,} / prot {len(prot_genes):,}")
print(f"  Gene universe     : {len(valid_ensg):,}")
print(f"  Expr in universe  : {len(expr_genes):,} ({len(expr_genes)/len(valid_ensg)*100:.1f}%)")
print(f"  Prot in universe  : {len(prot_genes):,} ({len(prot_genes)/len(valid_ensg)*100:.1f}%)")

print("\n[KEYS OK?]")
print(f"  expr index starts ACH-: {all(m.startswith('ach-') for m in list(expr_models)[:10])}")
print(f"  prot index starts ACH-: {all(m.startswith('ach-') for m in list(prot_models)[:10])}")
print(f"  expr cols start ENSG  : {all(c.startswith('ensg') for c in list(expr_genes)[:10])}")
print(f"  prot cols start ENSG  : {all(c.startswith('ensg') for c in list(prot_genes)[:10])}")
print(f"  any version suffix in expr cols: {any('.' in c for c in expr_genes)}")
print(f"  any version suffix in prot cols: {any('.' in c for c in prot_genes)}")

print("\n" + "=" * 60)
print("Confirm the above before running Cell 7 (scoring).")
print("=" * 60)

In [ ]:
# ── Cell 7 — Deduplicate + percentile normalisation per gene ──────────────
# RUN ONLY AFTER CELL 6 CONFIRMED.
#
# 16 ACHs have 2 RNA profiles each (dual-sequencing runs, confirmed in Cell 4).
# reindex() raises ValueError on duplicate index labels — must deduplicate first.
# Fix: mean across the two profiles for each duplicate ACH.
# Rationale: both profiles represent the same cell line; mean is the least
# lossy option (keeps magnitude; max would inflate, drop would lose data).
#
# Percentile rank (0-1) per gene, computed across all cell lines that have
# a measurement for that gene. NaN stays NaN (absent layer contributes no term).
#
# Why percentile not min-max:
#   - min-max is sensitive to outliers (audit confirmed weakest normalisation)
#   - percentile is robust by default; not a settled claim -- log/z-score are
#     also defensible and should be sensitivity-checked post-validation.

def dedup_mean(df):
    """Average duplicate index rows (same model_id, multiple RNA profiles)."""
    dups = df.index.duplicated(keep=False).sum()
    if dups:
        n_ach = df.index[df.index.duplicated(keep=False)].nunique()
        print(f"  Deduplicating {dups} rows ({n_ach} ACHs with >1 RNA profile) via mean")
        df = df.groupby(level=0).mean()
    return df

def percentile_rank(df):
    """Rank each column (gene) across rows (cell lines), map to [0,1].
    NaN values are excluded from ranking and stay NaN in output."""
    return df.rank(axis=0, pct=True, na_option="keep")

expr_dedup = dedup_mean(expr)
print(f"expr after dedup : {expr_dedup.shape}  (was {expr.shape})")
print(f"duplicate labels : {expr_dedup.index.duplicated().sum()}")

expr_pct = percentile_rank(expr_dedup)
prot_pct = percentile_rank(prot)

print(f"\nexpr_pct shape : {expr_pct.shape}")
print(f"prot_pct shape : {prot_pct.shape}")
print(f"expr_pct range : [{expr_pct.values.min():.3f}, {expr_pct.values.max():.3f}]")
print(f"prot_pct range : [{prot_pct.stack().min():.3f}, {prot_pct.stack().max():.3f}]")
print(f"expr_pct NaN % : {expr_pct.isna().mean().mean()*100:.1f}%")
print(f"prot_pct NaN % : {prot_pct.isna().mean().mean()*100:.1f}%")

In [ ]:
# -- Cell 7b - PATCH: tie handling in the percentile normalisation --
# Supersedes `percentile_rank` from Cell 7 and recomputes expr_pct / prot_pct.
# Original cell left intact for audit.
#
# Cell 7 used rank(pct=True) with pandas' default method="average", which gives
# every member of a tie group the MEAN rank of that group. For a layer where a
# large share of cell lines share the same value -- overwhelmingly "gene not
# expressed", value exactly 0 -- that block lands in the MIDDLE of the range it
# spans instead of at the floor:
#
#     gene    % lines tied at 0     percentile under "average"    under "min"
#     KLK4          56.7%                   0.2839                  0.0007
#     CD3E          38.1%                   0.1906                  0.0007
#     CD86          28.0%                   0.1405                  0.0007
#
# So a cell line with literally zero expression of KLK4 scored at the 28th
# percentile -- above every genuinely-low-but-nonzero line beneath it, and well
# above the floor it belongs on. The effect scales with how sparse the gene is,
# which means it is worst exactly for the tissue-restricted genes where absence
# is the informative signal.
#
# method="min" gives every member of a tie group the LOWEST rank in that group.
# Ties then carry no claim of superiority over each other, and a block of
# non-expressed lines sits at the bottom of the scale. Genes without large tie
# blocks are essentially unaffected (ERBB2's largest block moves 0.1609 ->
# 0.1599), and the best-scoring line still receives exactly 1.0.
#
# method="first" was rejected: it breaks ties by row order, making the score
# depend on how the matrix happened to be sorted.

def percentile_rank(df):
    """Rank each column (gene) across rows (cell lines), map to [0,1].

    Ties take the minimum rank of their group, so tied cell lines sit at the
    bottom of the range they span rather than its midpoint. NaN values are
    excluded from ranking and stay NaN in output."""
    return df.rank(axis=0, pct=True, na_option="keep", method="min")


expr_pct = percentile_rank(expr_dedup)
prot_pct = percentile_rank(prot)

print("Recomputed with method='min' (ties -> floor, not midpoint)")
print(f"  expr_pct shape : {expr_pct.shape}")
print(f"  prot_pct shape : {prot_pct.shape}")
print(f"  expr_pct range : [{expr_pct.values.min():.4f}, {expr_pct.values.max():.4f}]")
print(f"  prot_pct range : [{prot_pct.stack().min():.4f}, {prot_pct.stack().max():.4f}]")

_floor = (expr_pct <= 1.5 / len(expr_pct)).sum().sum()
print(f"  expr cells now sitting on the floor: {_floor:,} "
      f"({_floor / expr_pct.size * 100:.1f}% -- these are the non-expressed lines)")


In [ ]:
# -- Cell 7c - NEW: per-gene dynamic range (the "is this gene flat?" statistic) --
# Computed on the RAW pre-percentile expression, because the percentile transform
# in Cell 7/7b is exactly what destroys this information.
#
# THE PROBLEM THIS SOLVES
# core_score is a within-gene percentile rank. That rescales EVERY gene onto the
# same uniform 0-1 spread, so a housekeeping gene whose expression barely moves
# across cell lines produces a top-10 that looks every bit as confident as a
# sharply tissue-restricted gene's. GAPDH's best line scores 0.9993 and ERBB2's
# scores 1.0000 -- indistinguishable, despite GAPDH's top line being under 3x
# above its median and ERBB2's being ~75x. The ranking is not wrong so much as
# it is unable to say "there is nothing to rank here".
#
# THE STATISTIC
#   dynamic_range = p99 - p50 of raw log2(TPM+1) across cell lines
# i.e. how far above a typical line the top of the distribution actually sits,
# in log2 units, so it reads directly as fold-change. p99 rather than max so a
# single outlier line cannot manufacture range.
#
# WHY NOT THE OBVIOUS ALTERNATIVES (all checked against the real matrix):
#   IQR  -- inverted. Ranks GAPDH at the 79th percentile and CD3E at the 53rd,
#           because tissue-restricted genes are zero in most lines and therefore
#           have a NARROW interquartile range despite huge dynamic range. Using
#           IQR would have flagged exactly the wrong genes.
#   SD   -- muddled. GAPDH 0.700 vs KLK4 0.726: effectively tied.
#   CV   -- unusable. Mean approaches 0 for tissue-restricted genes, so CV
#           explodes (KLK4 3.46) for reasons unrelated to flatness.
#
# REFERENCE DISTRIBUTION
# Percentiles are taken against genes that are actually expressed (median
# log2TPM >= 1). Comparing against all ~54k columns is meaningless because over
# half are unexpressed and sit at dynamic_range ~0. Band cuts are derived from
# that reference at runtime, not hardcoded, so they track the matrix.
#
# VALIDATION (control genes chosen for their known biology, not fitted):
#   flat  : TBP 5%, TUBB 13%, GAPDH 19%, HPRT1 23%, RPL13A 27%, ACTB 34%
#   wide  : CD3E 99.8%, PTPRC 99.7%, MUC1 99%, ERBB2 99%, MITF 98%, ESR1 93%
#   BRAF sits at 25% -- correctly flat, since BRAF matters mutationally rather
#   than by abundance, which is precisely why it is driver-gated not score-ranked.
#
# This is a DESCRIPTIVE gene-level fact. It does not modify core_score and does
# not reorder anything -- it tells a reader whether the ordering is worth acting
# on. Folding it into the score would silently down-rank flat genes in a way no
# validation currently supports.

_raw = expr_dedup  # model_id x ensg_id, raw log2(TPM+1), pre-percentile

_p25, _p50, _p75, _p99 = (_raw.quantile(q) for q in (0.25, 0.50, 0.75, 0.99))

gene_dispersion = pd.DataFrame({
    "ensg_id":       _raw.columns,
    "n_lines":       _raw.notna().sum().to_numpy(),
    "median_log2tpm": _p50.to_numpy(),
    "iqr_log2tpm":   (_p75 - _p25).to_numpy(),
    "dynamic_range": (_p99 - _p50).to_numpy(),
})

# Reference = expressed genes only
_expressed = gene_dispersion[gene_dispersion["median_log2tpm"] >= 1.0]["dynamic_range"]
_lo, _hi = _expressed.quantile(0.25), _expressed.quantile(0.75)

gene_dispersion["dynamic_range_pct"] = (
    gene_dispersion["dynamic_range"].rank(pct=True, method="min")
)
gene_dispersion["signal_spread"] = pd.cut(
    gene_dispersion["dynamic_range"],
    bins=[-float("inf"), _lo, _hi, float("inf")],
    labels=["narrow", "moderate", "wide"],
).astype(str)

# Fold-change reading of the same number, for anyone not thinking in log2
gene_dispersion["top_vs_median_fold"] = (2 ** gene_dispersion["dynamic_range"]).round(2)

gene_dispersion.to_parquet(PIPELINE_OUT / "gene_dispersion.parquet", index=False)

print(f"gene_dispersion: {len(gene_dispersion):,} genes -> "
      f"{PIPELINE_OUT / 'gene_dispersion.parquet'}")
print(f"  reference: {len(_expressed):,} expressed genes (median log2TPM >= 1)")
print(f"  band cuts (derived, not hardcoded): narrow < {_lo:.3f} <= moderate <= {_hi:.3f} < wide")
print(f"  band sizes: {gene_dispersion['signal_spread'].value_counts().to_dict()}")
print(f"\n  {'gene':16s} {'median':>7s} {'dyn_range':>9s} {'fold':>7s}  band")
_gl = gene_lookup.set_index("ensg_id")["hgnc_symbol"]
for _sym in ["GAPDH", "ACTB", "TBP", "BRAF", "ERBB2", "MUC1", "CD3E"]:
    _e = _gl[_gl == _sym].index
    _r = gene_dispersion[gene_dispersion["ensg_id"].isin(_e)]
    if _r.empty:
        continue
    _r = _r.iloc[0]
    print(f"  {_sym:16s} {_r['median_log2tpm']:7.2f} {_r['dynamic_range']:9.2f} "
          f"{_r['top_vs_median_fold']:7.1f}x  {_r['signal_spread']}")


In [ ]:
# -- Cell 7c-silence - NEW: silence guard (frac_expressed) added to gene_dispersion --
# Extends Cell 7c. gene_dispersion's own "narrow" band does not catch most
# silent genes: of the 5,847 genes failing this guard, signal_spread calls
# 2,420 of them "wide", and 2,091 exceed FLAT_FOLD_CEILING=3.0 -- a dispersion
# check actively CERTIFIES these as well-dispersed rather than merely missing
# them (measured in docs/audit_scripts/d_f2.py PART F2; see
# docs/TRANSCRIPTOMICS_DECISIONS.md Q4). A gene can have a wide p99-p50 spread
# and still be silent (near-zero) in most lines with a long high tail; dynamic
# range does not see that, fraction-expressed does.
#
# EXPRESSED_MIN and SILENT_FRAC reuse the transcriptomics notebook's own
# constants unchanged (02_transcriptonomics.ipynb cell 1 /
# docs/audit_scripts/tx_lib.py), so the guard means the same thing in both
# places rather than being re-derived here.
#
# Consumed by evidence_state.py:gene_verdict, which folds
# `frac_expressed < SILENT_FRAC` into the same UNINFORMATIVE state as a
# narrow signal_spread (role separation C7 holds: this withholds/qualifies
# the ranking presented downstream, it does not touch core_score itself).

EXPRESSED_MIN = 1.0
SILENT_FRAC = 0.20

_frac_expressed = (_raw > EXPRESSED_MIN).sum() / _raw.notna().sum()
gene_dispersion["frac_expressed"] = gene_dispersion["ensg_id"].map(_frac_expressed)
gene_dispersion["silent_guard"] = gene_dispersion["frac_expressed"] < SILENT_FRAC

gene_dispersion.to_parquet(PIPELINE_OUT / "gene_dispersion.parquet", index=False)

_n_silent = int(gene_dispersion["silent_guard"].sum())
print(f"silence guard: {_n_silent:,} of {len(gene_dispersion):,} genes "
      f"({100*_n_silent/len(gene_dispersion):.1f}%) expressed (log2TPM > "
      f"{EXPRESSED_MIN}) in under {100*SILENT_FRAC:.0f}% of lines")
print("  overlap with signal_spread band (silent genes only):")
print(gene_dispersion.loc[gene_dispersion["silent_guard"], "signal_spread"]
      .value_counts().to_string())

In [ ]:
# -- Cell 7d - NEW: ProCan proteomics as a second protein platform --
# Adds data/proteomics_procan/Protein_matrix_averaged_20250211.tsv (DIA-MS,
# Goncalves et al.) alongside the CCLE/Gygi TMT matrix already loaded in Cell 5,
# and merges the two AT THE PERCENTILE LEVEL. Must run after Cell 7b, because it
# consumes and replaces `prot_pct`.
#
# WHY THIS MATTERS
# CCLE/Gygi covers 375 cell lines. The scored universe is ~1,480, so the protein
# layer reached only ~25% of lines and n_layers was 1 for the overwhelming
# majority of (gene, line) pairs -- the "multi-omics" score was single-omics
# nearly everywhere. ProCan covers 948 lines, 941 of which resolve to an ACH-
# model id through validation/prepared/id_bridge.parquet. Union coverage is
# roughly 1,025 lines (~69%), not 25%.
#
# WHY MERGING AT THE PERCENTILE LEVEL IS THE RIGHT JOIN
# test_run_proteomics_platform_overlap.py measured the two platforms against each
# other on their 291 shared lines and 6,121 comparable proteins: median per-
# protein Spearman rho 0.373, 26.7% of proteins above 0.5, 4.3% negative. Its
# verdict was "merge_with_caution -- merge only with per-dataset normalisation,
# and report the batch effect as a caveat".
#
# Percentile-ranking each platform separately IS that per-dataset normalisation,
# and it is already the pipeline's native representation: Cell 7b converts every
# layer to a within-gene rank across cell lines before anything is combined. A
# rank transform discards each platform's scale, dynamic range and batch offset
# entirely, so TMT log-ratios and DIA intensities never have to be made
# numerically comparable -- only their orderings are combined. Merging the raw
# matrices would have required exactly the cross-platform calibration the test
# said was unsafe.
#
# COMBINATION RULE
#   both platforms cover a (line, gene) -> mean of the two percentiles
#   one platform covers it            -> that platform's percentile
#   neither                            -> NaN (layer absent, n_layers unaffected)
# The mean is the conservative choice on the 291 shared lines: at rho 0.373 the
# platforms disagree materially, and averaging two noisy orderings is better
# behaved than trusting either. It is NOT evidence stacking -- this stays one
# protein layer, so n_layers is unchanged and a line measured twice does not
# score as having more evidence than a line measured once.
#
# CAVEAT TO CARRY INTO WRITE-UP: cross-platform proteomics agreement is only
# moderate. A protein layer sourced from ProCan is not interchangeable with one
# sourced from CCLE, and this cell does not pretend otherwise -- it makes the
# coverage honest, not the platforms equivalent.

import csv

_PROCAN = ROOT / "data" / "proteomics_procan" / "Protein_matrix_averaged_20250211.tsv"

# Layout: row0 = uniprot per column, row1 = gene symbol per column,
# row2 = ('model_name', 'model_id', '', ...), rows 3+ = one cell line each.
with open(_PROCAN, encoding="utf-8", errors="replace") as _fh:
    _r = csv.reader(_fh, delimiter="\t")
    _up_row = next(_r)
    next(_r)                      # symbol row -- we map via uniprot, as Cell 5 does
    next(_r)                      # header row
    _names, _sids, _vals = [], [], []
    for _row in _r:
        _names.append(_row[0])
        _sids.append(_row[1])
        _vals.append(_row[2:])

_uniprots = [u.strip().lower() for u in _up_row[2:]]
procan_raw = pd.DataFrame(_vals, columns=_uniprots, index=_sids)
procan_raw = procan_raw.replace("", np.nan).astype(float)
print(f"ProCan raw: {procan_raw.shape[0]} cell lines x {procan_raw.shape[1]} proteins")

# Sanger SIDM -> DepMap ACH-
_bridge = pd.read_parquet(ROOT / "validation" / "prepared" / "id_bridge.parquet")
_sid_to_ach = dict(zip(_bridge["sanger_model_id"].astype(str),
                       _bridge["model_id"].astype(str).str.lower()))
procan_raw.index = [_sid_to_ach.get(s) for s in procan_raw.index]
_unmapped = procan_raw.index.isna().sum() if hasattr(procan_raw.index, "isna") else \
            sum(1 for i in procan_raw.index if i is None)
procan_raw = procan_raw[[i is not None for i in procan_raw.index]]
procan_raw = procan_raw.groupby(level=0).mean()   # rule 1: dedupe after id mapping
print(f"  mapped to ACH-: {len(procan_raw)} lines ({_unmapped} unmapped, dropped)")

# UniProt -> ENSG using the patched Cell 3b map, then collapse to one col per gene
_u2e = dict(zip(uniprot_to_ensg["uniprot_id"], uniprot_to_ensg["ensg_id"]))
_keep = [c for c in procan_raw.columns if c in _u2e]
procan = procan_raw[_keep]
procan.columns = [_u2e[c] for c in _keep]
procan = procan.T.groupby(level=0).mean().T            # average duplicate accessions
procan = procan[[c for c in procan.columns if c in valid_ensg]]
print(f"  ENSG-mapped: {procan.shape[0]} lines x {procan.shape[1]} genes "
      f"({len(_keep)}/{procan_raw.shape[1]} protein columns resolved)")

# Independent percentile normalisation, then merge with the CCLE percentiles
procan_pct = percentile_rank(procan)

_ccle_pct = prot_pct
_all_m = sorted(set(_ccle_pct.index) | set(procan_pct.index))
_all_g = sorted(set(_ccle_pct.columns) | set(procan_pct.columns))
_A = _ccle_pct.reindex(index=_all_m, columns=_all_g)
_B = procan_pct.reindex(index=_all_m, columns=_all_g)

_both = _A.notna() & _B.notna()
prot_pct = _A.where(_A.notna(), _B)          # whichever exists
prot_pct[_both] = (_A[_both] + _B[_both]) / 2.0   # mean where both do

print(f"\nprotein layer coverage (cell lines with >=1 protein measurement):")
print(f"  CCLE/Gygi only : {_ccle_pct.notna().any(axis=1).sum():>5}")
print(f"  ProCan only    : {procan_pct.notna().any(axis=1).sum():>5}")
print(f"  merged union   : {prot_pct.notna().any(axis=1).sum():>5}  <- was CCLE-only before")
print(f"  overlap (mean-combined cells): {int(_both.sum().sum()):,}")
print(f"prot_pct shape: {prot_pct.shape}  range "
      f"[{prot_pct.stack().min():.4f}, {prot_pct.stack().max():.4f}]")


### Post-hoc correction — ProCan sourced from the harmonisation warehouse

`Cell 7d` bridges ProCan's Sanger `SIDM` ids to DepMap `ACH-` ids through
`validation/prepared/id_bridge.parquet`. That file has **no builder anywhere in
this repo**, so a clean rebuild cannot regenerate it, and it resolves 941 of the
948 ProCan lines — the other 7 are dropped with nothing recording which or why.

`00b_enriched_harmonisation.ipynb` resolves the same 948 lines through the
identity hub's Sanger axis and reaches **948 of 948**, because the hub is built
from two independent `SIDM → ACH` routes (Sanger's `BROAD_ID` and DepMap's
`sanger_model_id`, which were measured to agree on every line they share) rather
than from one unreproducible file.

The cell below re-derives the protein layer from `procan_proteomics` in
`outputs/celllineselector.db`. **Only the identity bridge changes** — the merge
rule, the normalisation and therefore the score semantics are identical.

In [ ]:
# -- Cell 7e - PATCH: source ProCan through the harmonisation warehouse --
# Supersedes the ProCan half of Cell 7d. Cell 7d is left intact and still runs
# first, so both resolutions are visible in one session and directly comparable.
#
# WHAT CHANGES -- only where the SIDM -> ACH bridge comes from:
#
#   Cell 7d  raw TSV + validation/prepared/id_bridge.parquet, a file with no
#            builder in this repo. 941/948 lines resolve, 7 dropped silently.
#   Cell 7e  `procan_proteomics` from outputs/celllineselector.db, resolved by
#            00b through the identity hub's Sanger axis. 948/948.
#
# The warehouse table is post-explode: a line whose SIDM resolved to two
# candidate ACHs appears once per candidate ("credit both"), flagged in
# is_ambiguous. Those are DIFFERENT model_ids, not duplicate rows, so both
# candidates are credited and nothing is collapsed away here.
#
# WHAT DOES NOT CHANGE -- the merge rule, and therefore the meaning of the score.
# Each platform is percentile-ranked independently; the mean is taken where both
# cover a (line, gene); a single platform is used where only one does. That is
# still the per-dataset normalisation test_run_proteomics_platform_overlap.py
# required (median per-protein rho 0.373 -> "merge only with per-dataset
# normalisation"), and this is still ONE protein layer, so n_layers is unaffected
# and a line measured on both platforms does not score as having more evidence.

import duckdb

_DB = PIPELINE_OUT / "celllineselector.db"
if not _DB.exists():
    raise FileNotFoundError(
        f"{_DB} not found. Run 00_harmonisation.ipynb, then "
        "00b_enriched_harmonisation.ipynb, before this notebook.")

_con = duckdb.connect(str(_DB), read_only=True)
_pc = _con.execute(
    "SELECT * FROM procan_proteomics WHERE model_id IS NOT NULL").df()
_con.close()

_META = {"gdsc_model_name", "sanger_model_id", "model_id",
         "matched_via", "n_model_id", "is_ambiguous"}
_ucols2 = [c for c in _pc.columns if c not in _META]

print(f"ProCan from warehouse : {len(_pc)} rows x {len(_ucols2):,} proteins")
print(f"  resolved via        : {_pc['matched_via'].value_counts().to_dict()}")
print(f"  rows credited to >1 ACH (ambiguous): {int(_pc['is_ambiguous'].sum())}")

# rule 1: model_id uppercase before any join (warehouse stores it lowercase)
procan2 = _pc[_ucols2].astype(float)
procan2.index = _pc["model_id"].str.lower().values
procan2 = procan2.groupby(level=0).mean()      # no-op unless one ACH repeats
print(f"  distinct ACH-       : {len(procan2)}")

# UniProt -> ENSG via the patched Cell 3b map. ProCan headers carry no isoform
# suffixes (checked: 0 of 8,453), so the alias extension Cell 3b needed for the
# CCLE matrix has no work to do here.
_u2e2 = dict(zip(uniprot_to_ensg["uniprot_id"], uniprot_to_ensg["ensg_id"]))
_keep2 = [c for c in procan2.columns if c in _u2e2]
procan2 = procan2[_keep2]
procan2.columns = [_u2e2[c] for c in _keep2]
procan2 = procan2.T.groupby(level=0).mean().T          # average duplicate accessions
procan2 = procan2[[c for c in procan2.columns if c in valid_ensg]]
print(f"  ENSG-mapped         : {procan2.shape[0]} lines x {procan2.shape[1]} genes "
      f"({len(_keep2)}/{len(_ucols2)} protein columns resolved)")

# Rebuild the merged protein layer from scratch. The CCLE percentiles are
# recomputed from `prot` (Cell 5, which Cell 7d never touches) rather than reused
# from Cell 7d's private _ccle_pct, so this cell is self-contained and correct
# whether or not 7d ran.
_prev_lines  = int(prot_pct.notna().any(axis=1).sum())   # what Cell 7d reached
_ccle_pct2   = percentile_rank(prot)
_procan_pct2 = percentile_rank(procan2)

_all_m2 = sorted(set(_ccle_pct2.index) | set(_procan_pct2.index))
_all_g2 = sorted(set(_ccle_pct2.columns) | set(_procan_pct2.columns))
_A2 = _ccle_pct2.reindex(index=_all_m2, columns=_all_g2)
_B2 = _procan_pct2.reindex(index=_all_m2, columns=_all_g2)

_both2 = _A2.notna() & _B2.notna()
prot_pct = _A2.where(_A2.notna(), _B2)                 # whichever exists
prot_pct[_both2] = (_A2[_both2] + _B2[_both2]) / 2.0   # mean where both do

print(f"\nprotein layer coverage (cell lines with >=1 protein measurement):")
print(f"  CCLE/Gygi only : {_ccle_pct2.notna().any(axis=1).sum():>5}")
print(f"  ProCan only    : {_procan_pct2.notna().any(axis=1).sum():>5}")
print(f"  merged union   : {prot_pct.notna().any(axis=1).sum():>5}"
      f"   (Cell 7d reached {_prev_lines})")
print(f"  overlap (mean-combined cells): {int(_both2.sum().sum()):,}")
print(f"prot_pct shape: {prot_pct.shape}  range "
      f"[{prot_pct.stack().min():.4f}, {prot_pct.stack().max():.4f}]")

assert prot_pct.index.is_unique, "duplicate model_id in the merged protein layer"
assert prot_pct.notna().any(axis=1).sum() >= _prev_lines, \
    "warehouse-sourced ProCan reached FEWER lines than the id_bridge route"

In [ ]:
# -- Cell 7f - PATCH: protein-platform conflict tier + abundance filter --
# Supersedes the naive-mean blend in Cell 7e for conflicting/low-abundance proteins.
#
# DECISIONS IMPLEMENTED:
#   Decision 1 -- Apply protein tier filter
#     For the 36.7% of proteins in the conflicting tier (rho < 0.3 between
#     ProCan and CCLE/Gygi), stop blending. Use ProCan alone (preferred:
#     more lines, DIA-MS absolute quantification).
#   Decision 2 -- Combine tier + abundance quartile
#     Restrict the mean-blend to proteins that are BOTH consistent/cautious
#     tier AND Q3/Q4 abundance (top half of ProCan median intensity). Bottom-
#     half proteins (Q1/Q2) are reverted to single-platform even if in the
#     cautious tier: Step 6 showed Q1 median rho = 0.235, below safe-merge
#     threshold. Q4 (high-abundance) reached rho = 0.511.
#
# BLEND RULE after this patch:
#   (consistent OR cautious) AND Q3/Q4 abundance -> mean of both percentiles
#   conflicting tier OR Q1/Q2 abundance          -> ProCan percentile only
#
# Variables in scope from Cell 7e: _u2e2, _both2, _B2, prot_pct, procan2.

from pathlib import Path as _Path

# Resolve tier parquet regardless of Jupyter CWD
_TIER_PATH = next(
    (p for p in [
        _Path("outputs/protein_platform_tier.parquet"),
        _Path("src/pipeline/outputs/protein_platform_tier.parquet"),
    ] if p.exists()),
    None,
)
if _TIER_PATH is None:
    raise FileNotFoundError(
        "protein_platform_tier.parquet not found. "
        "Run test_run_proteomics_platform_overlap.py first.")

_tier_df = pd.read_parquet(_TIER_PATH)   # columns: uniprot, platform_rho, platform_tier
_tier_df["ensg_id"] = _tier_df["uniprot"].map(_u2e2)
_tier_df = _tier_df.dropna(subset=["ensg_id"])

# Decision 1: proteins where the two platforms actively disagree
_conflicting_ensg = set(
    _tier_df.loc[_tier_df["platform_tier"] == "conflicting", "ensg_id"])

# Decision 2: bottom-half abundance -- ProCan median intensity < 50th percentile
# (Q1/Q2 from Step 6; Q3 starts at +2.91 in the test run)
_prot_med = procan2.median(axis=0)      # per-gene median across ProCan lines
_q2q3_cutoff = float(_prot_med.quantile(0.50))
_low_abund_ensg = set(_prot_med[_prot_med < _q2q3_cutoff].index)

# Combined: revert to single-platform if conflicting tier OR low abundance
_exclude_ensg = _conflicting_ensg | _low_abund_ensg

# Build aligned masks (same layout as prot_pct from Cell 7e)
_excl_series = pd.Series(
    prot_pct.columns.isin(_exclude_ensg), index=prot_pct.columns, dtype=bool)
_both2_aligned = _both2.reindex(
    index=prot_pct.index, columns=prot_pct.columns).fillna(False)
_B2_aligned = _B2.reindex(index=prot_pct.index, columns=prot_pct.columns)

# Where both platforms had data AND protein is in the exclusion set:
# revert from the Cell-7e mean-blend to ProCan alone.
_revert = _both2_aligned & _excl_series   # broadcasts _excl_series across rows
prot_pct = prot_pct.where(~_revert, _B2_aligned)

_n_reverted = int(_revert.values.sum())
_n_blended  = int((_both2_aligned & ~_excl_series).values.sum())

print("Protein tier + abundance filter (Cell 7f):")
print(f"  conflicting-tier proteins (rho < 0.3)       : {len(_conflicting_ensg):,} ENSG IDs")
print(f"  low-abundance proteins (Q1/Q2, <{_q2q3_cutoff:.2f})     : {len(_low_abund_ensg):,} ENSG IDs")
print(f"  combined exclusion from blend               : {len(_exclude_ensg):,} ENSG IDs")
print(f"  shared cells reverted to ProCan alone       : {_n_reverted:,}")
print(f"  shared cells still mean-blended (safe zone) : {_n_blended:,}")
print(f"prot_pct shape            : {prot_pct.shape}")
print(f"prot_pct NaN fraction     : {prot_pct.isna().mean().mean()*100:.1f}%")


-- Cell 7g -- NEW: Multi-source RNA expression layer (replaces DepMap-only expr_pct)

Replaces `expr_pct` (DepMap-only percentile) with a percentile derived from
lineage-conditioned z-scores combining DepMap + HPA + warehouse GEO.

**Why lineage-conditioned z-scores beat raw percentile here:**
The DepMap-only percentile ranks every cell line globally for a gene, which
means a kidney line with typical kidney expression looks identical to a skin
line that's genuinely elevated above its peers. The lineage-conditioned z-score
`rna_z_t` already encodes 'high relative to lineage', so converting it back to
a global percentile preserves that ordering without losing the cross-gene
comparability that the Noisy-OR needs.

**Coverage gain vs DepMap-only:**
- DepMap only: 1,479 model_ids
- DepMap + HPA + GEO: 1,501 model_ids (HPA alone adds ~24 lines not in DepMap)
- n_sources column: 1 = only one source measured; 2 = two sources agree;
  3 = all three sources measured (use `n_sources` to flag single-source uncertainty)


In [ ]:
# -- Cell 7g -- Multi-source RNA z -> percentile, replaces DepMap-only expr_pct --
# Run expression_fusion/05_bulk_rna_scorer.py first to produce bulk_rna_z.parquet.
# That script loads all three sources in ~7 minutes and pre-computes lineage-
# conditioned robust z-scores for all 19,899 genes.

import pathlib as _pl
_bulk_path = _pl.Path(r"C:\\Disertation\\UoB-GeneTraceAI-25-26\\expression_fusion\\outputs\\bulk_rna_z.parquet\")\n
if not _bulk_path.exists():
    print("bulk_rna_z.parquet not found -- falling back to DepMap-only expr_pct.")
    print("Run: cd expression_fusion && python 05_bulk_rna_scorer.py")
else:
    _rna_z = pd.read_parquet(_bulk_path)
    # pivot to wide: model_id (index) x gene_id (columns)
    # ENSG ids must match the ensg_id key used in the rest of this notebook
    _rna_z["ensg_id"] = _rna_z["gene_id"].str.upper()   # normalise to UPPER

    _rna_wide = _rna_z.pivot(
        index="model_id", columns="ensg_id", values="rna_z_t"
    )
    # model_id normalisation: core score uses UPPER ACH- ids
    _rna_wide.index = _rna_wide.index.str.upper()

    # Convert lineage-conditioned z -> global percentile per gene.
    # The ordering is preserved: a line that is high relative to its lineage
    # peers will rank highly globally. method='min' matches the tie-handling
    # fix in Cell 7b that pushes tied-zero genes to the floor.
    # Keep z-scores as rna_z_wide -- the orthogonal combination in Cell 7i
    # uses these directly. expr_pct kept for reference only (unused in scoring).
    rna_z_wide = _rna_wide.copy()
    expr_pct   = _rna_wide.rank(pct=True, method="min")  # reference; not fed to Cell 8

    # Also store the n_sources pivot so downstream cells can split by evidence depth
    _nsrc_wide = _rna_z.pivot(
        index="model_id", columns="ensg_id", values="n_sources"
    )
    _nsrc_wide.index = _nsrc_wide.index.str.upper()
    rna_n_sources = _nsrc_wide   # available for audit; not used in scoring

    print(f"expr_pct replaced with multi-source RNA z-score percentile")
    print(f"  shape:     {expr_pct.shape}")
    print(f"  model_ids: {expr_pct.shape[0]:,}  (was {len(expr_models):,} DepMap-only)")
    print(f"  genes:     {expr_pct.shape[1]:,}")
    print(f"  n_sources 1/2/3: {int((_rna_z.n_sources==1).sum()):,} / "
          f"{int((_rna_z.n_sources==2).sum()):,} / "
          f"{int((_rna_z.n_sources==3).sum()):,}")

-- Cell 7h -- Load bulk protein z-scores (orthogonal combination input)

Loads `bulk_prot_z.parquet` produced by
`expression_fusion/06_bulk_protein_scorer.py`. Columns are lineage-
conditioned Stouffer-combined z-scores from ProCAN (DIA-MS) + CCLE (TMT),
normalised within each source and lineage before combining — the same
approach as the RNA scorer.

In [ ]:
# -- Cell 7h -- Protein z-scores from bulk scorer
import pathlib as _pl
_prot_z_path = _pl.Path(r"C:\Disertation\UoB-GeneTraceAI-25-26\expression_fusion\outputs\bulk_prot_z.parquet")

if not _prot_z_path.exists():
    print("bulk_prot_z.parquet not found -- protein layer will be skipped.")
    print("Run: cd expression_fusion && python 06_bulk_protein_scorer.py")
    prot_z_wide = None
else:
    _prot_z = pd.read_parquet(_prot_z_path)
    _prot_z["ensg_id"] = _prot_z["gene_id"].str.lower()  # already lower; belt+braces
    _prot_z_pivot = _prot_z.pivot(
        index="model_id", columns="ensg_id", values="prot_z_t"
    )
    _prot_z_pivot.index = _prot_z_pivot.index.str.upper()  # UPPER ACH- to match core score
    prot_z_wide = _prot_z_pivot

    # n_sources audit (1=ProCAN-only or CCLE-only, 2=both agreed)
    _prot_nsrc = _prot_z.pivot(
        index="model_id", columns="ensg_id", values="n_sources"
    )
    _prot_nsrc.index = _prot_nsrc.index.str.upper()
    prot_n_sources = _prot_nsrc

    print(f"prot_z_wide loaded: {prot_z_wide.shape}")
    print(f"  model_ids: {prot_z_wide.shape[0]:,}")
    print(f"  genes:     {prot_z_wide.shape[1]:,}")
    print(f"  n_sources 1={int((_prot_z.n_sources==1).sum()):,}  "
          f"2={int((_prot_z.n_sources==2).sum()):,}")

-- Cell 7i -- Orthogonal RNA-protein combination

Replaces Noisy-OR (Cell 8) and correlation-penalised mean (Cell 8b).

**Why orthogonalise instead of plain weighted sum?**
RNA and protein are correlated (rho ~ 0.37-0.46). A weighted sum treats
them as independent evidence, inflating the combined score. Partial
residualisation removes the RNA-explained variance from the protein signal
before adding it -- the remaining protein contribution is genuinely new
information (post-transcriptional regulation, protein stability, etc.).

**Combination formula** for cells with both RNA and protein:
```
prot_resid_z = (prot_z - rho_g * rna_z) / sqrt(1 - rho_g^2)
core_z = (W_RNA * rna_z + W_PROT * prot_resid_z) / sqrt(W_RNA^2 + W_PROT^2)
core_score = Phi(core_z)   # standard normal CDF -> (0, 1)
```
where rho_g is estimated per gene from overlapping lines (shrunk toward
the global prior of 0.373 when fewer than 30 lines overlap).

**Single-source cells:** `core_z = rna_z_t` (RNA only) or `prot_z_t`
(protein only). `n_layers` (0/1/2) is set accordingly for `stratum_rank`.

In [ ]:
# -- Cell 7i -- Orthogonal RNA-protein combination -> core_score
from scipy import stats as _scipy_stats

# Kish-corrected layer weights
# RNA: W_RNA is PROVISIONAL. n_eff=2.0 requires rho_bar=0.25 across all 3
# source pairs, but DepMap-HPA alone is 0.812, so rho_bar >= 0.25 is impossible.
# True n_eff is estimated < 1.60 (see EXPR_FUSION_PREREGISTRATION.md T3).
# Replace sqrt(2.00) with sqrt(n_eff_actual) once T3 has been run.
# Protein: 2 sources, ProCAN-CCLE rho=0.373 -> n_eff = 2/(1+0.373) = 1.45
_W_RNA  = np.sqrt(2.00)   # PROVISIONAL — awaiting T3 correction
_W_PROT = np.sqrt(1.45)   # 1.204
_W_NORM = np.sqrt(_W_RNA**2 + _W_PROT**2)

_RHO_PRIOR = 0.373   # median ProCAN-CCLE Spearman rho from overlap test
_N_MIN_RHO = 30      # shrink toward prior below this overlap count

# ── align on common axes ──────────────────────────────────────────────────
_rna = rna_z_wide  # model_id (UPPER) x ensg_id (lower)

if prot_z_wide is None:
    # Protein scorer not run: RNA-only score
    print("WARNING: prot_z_wide is None. Scoring RNA layer only.")
    _all_models = sorted(_rna.index)
    _all_genes  = sorted(_rna.columns)
    core_z      = _rna.reindex(index=_all_models, columns=_all_genes)
    n_layers    = (~core_z.isna()).astype(int)
    rho_audit   = pd.Series(_RHO_PRIOR, index=_all_genes, name="rho_g")
else:
    _prot = prot_z_wide

    _all_models = sorted(set(_rna.index) | set(_prot.index))
    _all_genes  = sorted(set(_rna.columns) | set(_prot.columns))

    _R = _rna.reindex(index=_all_models, columns=_all_genes)   # NaN where absent
    _P = _prot.reindex(index=_all_models, columns=_all_genes)

    # ── per-gene rho estimation ───────────────────────────────────────────
    # Only computable for genes in both layers; use prior for RNA-only genes.
    _common_genes   = _rna.columns.intersection(_prot.columns)
    _overlap_models = _rna.index.intersection(_prot.index)

    _r_ov = _R.loc[_overlap_models, _common_genes].values.astype(float)  # (n_ov, n_cg)
    _p_ov = _P.loc[_overlap_models, _common_genes].values.astype(float)

    _both_valid = np.isfinite(_r_ov) & np.isfinite(_p_ov)
    _n_valid    = _both_valid.sum(axis=0)  # per gene

    # Pearson r per gene, vectorised
    _r_m = np.where(_both_valid, _r_ov, np.nan)
    _p_m = np.where(_both_valid, _p_ov, np.nan)
    _r_c = _r_m - np.nanmean(_r_m, axis=0)
    _p_c = _p_m - np.nanmean(_p_m, axis=0)
    _num  = np.nansum(_r_c * _p_c,     axis=0)
    _den  = np.sqrt(np.nansum(_r_c**2, axis=0) * np.nansum(_p_c**2, axis=0))
    _rho_raw = np.where(_den > 0, _num / np.where(_den > 0, _den, 1.0), np.nan)

    # James-Stein shrinkage toward prior
    _lam     = np.where(_n_valid >= _N_MIN_RHO,
                        _N_MIN_RHO / (_n_valid + _N_MIN_RHO), 1.0)
    _rho_cg  = _lam * _RHO_PRIOR + (1 - _lam) * np.where(np.isfinite(_rho_raw),
                                                           _rho_raw, _RHO_PRIOR)
    _rho_cg  = np.clip(_rho_cg, -0.99, 0.99)

    # Broadcast rho to all_genes (RNA-only genes get the prior)
    rho_audit = pd.Series(_RHO_PRIOR, index=_all_genes, name="rho_g")
    rho_audit[_common_genes] = _rho_cg

    print(f"Per-gene rho: mean={rho_audit.mean():.3f}  "
          f"median={rho_audit.median():.3f}  "
          f"std={rho_audit.std():.3f}")
    print(f"Genes with n_overlap >= {_N_MIN_RHO}: "
          f"{int((_n_valid >= _N_MIN_RHO).sum()):,} / {len(_common_genes):,}")

    # ── orthogonal combination ────────────────────────────────────────────
    # Align rho to gene axis of _R, _P
    _rho_arr = rho_audit.reindex(_all_genes).values  # (n_genes,)

    _R_vals = _R.values  # (n_models, n_genes)
    _P_vals = _P.values

    # Residual: protein signal orthogonal to RNA
    _prot_resid    = _P_vals - _rho_arr[np.newaxis, :] * _R_vals
    _resid_scale   = np.sqrt(1.0 - _rho_arr**2)[np.newaxis, :]  # (1, n_genes)
    _resid_scale   = np.where(_resid_scale > 0, _resid_scale, 1.0)
    _prot_resid_z  = _prot_resid / _resid_scale

    # Weighted combination where BOTH layers present
    _rna_present  = np.isfinite(_R_vals)
    _prot_present = np.isfinite(_P_vals)
    _both_present = _rna_present & _prot_present

    _core_z = np.full_like(_R_vals, np.nan)

    # Both: orthogonal weighted combination
    _core_z[_both_present] = (
        _W_RNA  * _R_vals[_both_present]
        + _W_PROT * _prot_resid_z[_both_present]
    ) / _W_NORM

    # RNA only
    _rna_only = _rna_present & ~_prot_present
    _core_z[_rna_only] = _R_vals[_rna_only]

    # Protein only (rare: a model_id in protein but not RNA)
    _prot_only = ~_rna_present & _prot_present
    _core_z[_prot_only] = _P_vals[_prot_only]

    core_z   = pd.DataFrame(_core_z, index=_all_models, columns=_all_genes)
    n_layers = _rna_present.astype(int) + _prot_present.astype(int)
    n_layers = pd.DataFrame(n_layers, index=_all_models, columns=_all_genes)

# ── map z-scores to (0, 1) via standard normal CDF ───────────────────────
# Phi(z) preserves all magnitude information: z=4 -> 0.99997, z=1 -> 0.841.
# Unlike rank(pct=True), two lines at z=4 vs z=1 are NOT equidistant.
core_score = core_z.apply(
    lambda col: _scipy_stats.norm.cdf(col.values), axis=0
)
core_score = pd.DataFrame(
    core_score, index=core_z.index, columns=core_z.columns
)

# Propagate NaN (no evidence) through Phi
core_score[core_z.isna()] = np.nan

_n_scored = core_score.notna().sum().sum()
_n_total  = core_score.size
print(f"\ncore_score shape  : {core_score.shape}")
print(f"non-NaN cells     : {_n_scored:,} / {_n_total:,} "
      f"({_n_scored/_n_total*100:.1f}%)")
print(f"score range       : [{core_score.stack().min():.4f}, "
      f"{core_score.stack().max():.4f}]")
print(f"n_layers distribution:")
print(n_layers.stack().value_counts().sort_index().to_string())

In [ ]:
# ── Cell 9 — Melt to long form + stratum-aware rank ───────────────────────
# (1) DONE: stratum_rank ranks cell lines within each (ensg_id, n_layers) group.
#     Do NOT sort a merged list by core_score across strata without this flag.
#     A line with n_layers=2 and core_score=0.7 is not comparable to
#     a line with n_layers=1 and core_score=0.7 — different evidence bases.
#
# (2) DONE: n_layers is always present alongside core_score in every output row.
#     A reader ranking cell lines sees "score + stratum" together, not score alone.
#
# (3) HARNESS-GATED — NOT BUILT YET:
#     Cross-stratum calibration (making a 1-layer score and a 2-layer score
#     numerically comparable) requires a held-out regression / correlation-
#     correction against CRISPR/GDSC anchor pairs. Deferred until the
#     validation harness exists. Flag in outputs: n_layers tells you which
#     stratum a score belongs to; stratum_rank is the safe within-stratum rank.

score_long = (
    core_score
    .stack()
    .reset_index()
    .rename(columns={"level_0": "model_id", "level_1": "ensg_id", 0: "core_score"})
)

nlayers_long = (
    n_layers
    .stack()
    .reset_index()
    .rename(columns={"level_0": "model_id", "level_1": "ensg_id", 0: "n_layers"})
)

before = len(score_long)
score_long = score_long.merge(nlayers_long, on=["model_id", "ensg_id"], how="left")
assert len(score_long) == before, "Fan-out on n_layers merge"

score_long = score_long.dropna(subset=["core_score"])
score_long["n_layers"] = score_long["n_layers"].astype(int)

# (1) Stratum-aware rank: percentile rank of core_score within (ensg_id, n_layers).
#     Use this when comparing cell lines for the same gene.
#     Do NOT use raw core_score for cross-stratum comparisons.
score_long["stratum_rank"] = (
    score_long
    .groupby(["ensg_id", "n_layers"])["core_score"]
    .rank(pct=True, method="average")
)

print(f"score_long shape     : {score_long.shape}")
print(f"columns              : {score_long.columns.tolist()}")
print(f"distinct model_ids   : {score_long['model_id'].nunique():,}")
print(f"distinct ensg_ids    : {score_long['ensg_id'].nunique():,}")
print(f"\nn_layers distribution:")
print(score_long["n_layers"].value_counts().sort_index().to_string())
print(f"\nstratum_rank range   : [{score_long['stratum_rank'].min():.4f}, {score_long['stratum_rank'].max():.4f}]")
print(f"\nSample rows:")
print(score_long.head(5).to_string(index=False))


In [ ]:
# -- Cell 9b - PATCH: same tie rule for stratum_rank --
# Supersedes the stratum_rank computed in Cell 9. Original left intact.
#
# stratum_rank re-ranks core_score within each (ensg_id, n_layers) group. It was
# using method="average", which would undo Cell 7b: a block of cell lines pushed
# to the floor of core_score would be re-inflated back to the midpoint of its tie
# range as soon as it was expressed as a stratum percentile. The two ranks have
# to agree on how ties are treated or the fix only holds in one of the two
# columns the ranking layer exposes.

score_long["stratum_rank"] = (
    score_long
    .groupby(["ensg_id", "n_layers"])["core_score"]
    .rank(pct=True, method="min")
)

print("stratum_rank recomputed with method='min'")
print(f"  range: [{score_long['stratum_rank'].min():.4f}, {score_long['stratum_rank'].max():.4f}]")
_agree = (score_long.groupby("ensg_id")
          .apply(lambda d: d["core_score"].corr(d["stratum_rank"], method="spearman"),
                 include_groups=False)
          .dropna())
print(f"  median per-gene Spearman(core_score, stratum_rank): {_agree.median():.4f} "
      f"(monotone within stratum by construction)")


In [ ]:
# ── Cell 10 — Spot-check: anchor gene × anchor cell line ──────────────────
# RRID-based resolver: name → CVCL (Cellosaurus) → model_id (harmonised_enriched rrids)
# Deterministic — no substring collisions, no silent first-hit errors.
# Audit (Cell 10) quantifies improvement vs old fuzzy approach.

import json, re

he = pd.read_parquet(PIPELINE_OUT / "harmonised_enriched.parquet")

def norm(s):
    return re.sub(r"[\s\-_\.]", "", str(s)).lower()

# Build name → {CVCL, ...} from Cellosaurus (primary name + semicolon-separated synonyms)
cel = pd.read_parquet(ROOT / "data" / "parquet" / "data_clean" / "cellosaurus_clean.parquet",
                      columns=["cellosaurus_cell_line_name", "cellosaurus_accession", "synonyms"])
name_to_cvcls = {}
for _, r in cel.iterrows():
    cvcl = str(r["cellosaurus_accession"]).strip()
    candidates = [r["cellosaurus_cell_line_name"]] + (
        str(r["synonyms"]).split(";") if pd.notna(r["synonyms"]) else []
    )
    for raw in candidates:
        key = norm(raw)
        if key:
            name_to_cvcls.setdefault(key, set()).add(cvcl)

# Build CVCL → model_id from rrids column (JSON string per row)
cvcl_to_model = {}
for _, row in he.iterrows():
    try:
        rrids = json.loads(row["rrids"]) if isinstance(row["rrids"], str) else []
    except Exception:
        rrids = []
    mid = str(row["model_id"]).lower()
    for r in rrids:
        cvcl_to_model[r.strip()] = mid

def model_id_for(name):
    """Resolve cell line name -> ACH- via Cellosaurus CVCL. Returns (model_id, reason)."""
    key = norm(name)
    cvcls = name_to_cvcls.get(key)
    if not cvcls:
        return None, "name_not_in_cellosaurus"
    mids = list({cvcl_to_model[c] for c in cvcls if c in cvcl_to_model})
    if not mids:
        return None, "cvcl_not_in_harmonised"
    if len(mids) > 1:
        return None, "multi_model_collision"
    return mids[0], "ok"

ANCHORS = {
    "BRAF": "ensg00000157764",
    "KRAS": "ensg00000133703",
    "EGFR": "ensg00000146648",
}

anchor_cells = {"A375": model_id_for("A375"), "HCT116": model_id_for("HCT116")}
print("Anchor cell line model_ids:")
for name, (mid, reason) in anchor_cells.items():
    print(f"  {name}: {mid}  ({reason})")

print("\nAnchor scores:")
for gene_name, ensg in ANCHORS.items():
    for cell_name, (mid, _) in anchor_cells.items():
        if mid is None:
            print(f"  {gene_name}/{cell_name}: model_id not resolved")
            continue
        row = score_long[(score_long["model_id"] == mid) & (score_long["ensg_id"] == ensg)]
        if len(row):
            s = row.iloc[0]
            print(f"  {gene_name}/{cell_name}: core_score={s['core_score']:.4f}  n_layers={int(s['n_layers'])}")
        else:
            print(f"  {gene_name}/{cell_name}: NO DATA")

In [ ]:
# ── Audit — RRID vs fuzzy name matching ───────────────────────────────────
# Results from pre-run: 18 RRID_RESCUED, 9 OK, 1 WRONG (fuzzy wrong on A375
# derivative), 1 BOTH_MISSING (LNCaP not in DepMap), 1 FUZZY_ONLY (PC3 collision).

TEST_NAMES = [
    "A375","HCT116","MCF7","MCF10A","MDA-MB-231","MDA-MB-468","PC3","LNCaP",
    "DU145","HeLa","K562","HL60","Jurkat","THP1","U87MG","U251","T47D",
    "BT474","SKBR3","HT29","SW480","SW620","A549","H1299","H460","PANC1",
    "MiaPaCa2","HepG2","Huh7","Caco2"
]

def fuzzy_model_id_for(fragment):
    mask = he["canonical_name"].astype("string").str.lower().str.contains(
        fragment.lower(), na=False, regex=False)
    hits = he.loc[mask, ["model_id", "canonical_name"]]
    return (hits.iloc[0]["model_id"].lower() if len(hits) > 0 else None), len(hits)

rows = []
for name in TEST_NAMES:
    fuzzy_mid, n_fuzzy = fuzzy_model_id_for(name)
    rrid_mid, rrid_reason = model_id_for(name)
    if fuzzy_mid is None and rrid_mid is None:
        verdict = "BOTH_MISSING"
    elif fuzzy_mid is None and rrid_mid is not None:
        verdict = "RRID_RESCUED"
    elif fuzzy_mid is not None and rrid_mid is None:
        verdict = "FUZZY_ONLY"
    elif fuzzy_mid != rrid_mid:
        verdict = "WRONG"
    elif n_fuzzy > 1:
        verdict = "AMBIGUOUS_BUT_AGREE"
    else:
        verdict = "OK"
    rows.append({"name": name, "fuzzy_result": fuzzy_mid, "rrid_result": rrid_mid,
                 "rrid_reason": rrid_reason, "n_fuzzy_hits": n_fuzzy, "verdict": verdict})

audit = pd.DataFrame(rows)
print("=== Verdict counts ===")
print(audit["verdict"].value_counts().to_string())
for v in ["WRONG", "FUZZY_ONLY"]:
    sub = audit[audit["verdict"] == v]
    if len(sub):
        print(f"\n=== {v} ===")
        print(sub[["name","fuzzy_result","rrid_result","rrid_reason","n_fuzzy_hits"]].to_string(index=False))
audit.to_parquet(PIPELINE_OUT / "name_match_audit_v2.parquet", index=False)
print(f"\nSaved -> {PIPELINE_OUT}/name_match_audit_v2.parquet")
audit

In [ ]:
# ── Cell 11 — Write output ────────────────────────────────────────────────
# Schema: (model_id, ensg_id, core_score, n_layers, stratum_rank)
#
# HOW TO USE CORRECTLY:
#   - Rank cell lines for a gene: use stratum_rank, filter to ONE n_layers value.
#     e.g. top lines for BRAF in the 2-layer stratum:
#       score_long[(score_long.ensg_id=='ensg00000157764') & (score_long.n_layers==2)]
#             .sort_values('stratum_rank', ascending=False)
#   - Cross-gene comparison within a cell line: core_score is valid (same cell line,
#     same coverage stratum by definition).
#   - Cross-stratum comparison: NOT safe with current build. See item (3) below.
#
# ITEM (3) — HARNESS-GATED, NOT BUILT:
#   Cross-stratum calibration would make a 1-layer score and a 2-layer score
#   numerically comparable. Requires held-out regression against CRISPR/GDSC
#   anchor pairs (BRAF/A375, KRAS/HCT116 + TSG-fusion arm). Deferred.
#   When it exists, replace stratum_rank with a calibrated_score column and
#   drop the cross-stratum restriction.

OUT_PATH = PIPELINE_OUT / "core_score.parquet"
score_long.to_parquet(OUT_PATH, index=False)

print(f"Written : {OUT_PATH}")
print(f"Shape   : {score_long.shape}")
print(f"Columns : {score_long.columns.tolist()}")
print(f"\n[RANKING RULES]")
print(f"  Within-stratum rank : use stratum_rank, filter n_layers first")
print(f"  Cross-gene (same line): use core_score directly")
print(f"  Cross-stratum        : NOT SAFE — harness-gated (item 3)")
print(f"\n[CALIBRATION STATUS]")
print(f"  Coefficients are uncalibrated free parameters.")
print(f"  Validation against CRISPR/GDSC anchors has not been run.")
